# Sistema Integral de Gestión de Clientes, Servicios y Reservas
## Empresa: Software FJ

**Universidad Nacional Abierta y a Distancia – UNAD**  
**Escuela de Ciencias Básicas, Tecnología e Ingeniería – ECBTI**  
**Curso:** Programación (213023)  
**Fase 4 – Componente práctico - Prácticas simuladas**

---

Este notebook implementa una aplicación orientada a objetos para la empresa
**Software FJ**, capaz de gestionar **clientes**, **servicios** (reservas de
salas, alquiler de equipos y asesorías especializadas) y **reservas**, sin
utilizar bases de datos.

### Principios POO aplicados
- **Abstracción** → clases abstractas `EntidadBase` y `Servicio`
- **Herencia** → `Cliente`, `ReservaSala`, `AlquilerEquipo`, `AsesoriaTecnica`
- **Polimorfismo** → métodos `calcular_costo()` y `descripcion()` sobrescritos
- **Encapsulación** → atributos privados con *properties*
- **Sobrecarga** → `calcular_costo_total()` con parámetros opcionales

### Manejo avanzado de excepciones
- Excepciones personalizadas (`SoftwareFJError` y derivadas)
- Bloques `try/except`, `try/except/else`, `try/except/finally`
- Encadenamiento de excepciones con `raise ... from ...`
- Registro de eventos y errores en archivo `software_fj.log`


## 1. Importaciones y configuración del sistema de logs

In [1]:
# Importación de módulos estándar de Python
import re                       # Validar formatos con expresiones regulares
import logging                  # Sistema de registro (logs)
from abc import ABC, abstractmethod  # Clases y métodos abstractos
from datetime import datetime   # Fechas y horas de eventos

# Configuración del sistema de logs.
# Todos los eventos relevantes se almacenan en "software_fj.log".
logging.basicConfig(
    filename="software_fj.log",
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    encoding="utf-8"
)
logger = logging.getLogger("SoftwareFJ")
print("Sistema de logging configurado correctamente.")

Sistema de logging configurado correctamente.


## 2. Excepciones personalizadas

Jerarquía de excepciones propias del dominio de Software FJ.
Todas heredan de `SoftwareFJError` para permitir capturarlas de forma agrupada
o individual.

In [2]:
class SoftwareFJError(Exception):
    """Clase base para todas las excepciones del sistema Software FJ."""
    pass


class DatoInvalidoError(SoftwareFJError):
    """Se lanza cuando un dato no cumple las reglas de validación."""
    pass


class ParametroFaltanteError(SoftwareFJError):
    """Se lanza cuando un parámetro obligatorio no fue proporcionado."""
    pass


class ServicioNoDisponibleError(SoftwareFJError):
    """Se lanza cuando se intenta reservar un servicio no disponible."""
    pass


class ReservaInvalidaError(SoftwareFJError):
    """Se lanza cuando se intenta crear una reserva con datos inconsistentes."""
    pass


class CalculoInconsistenteError(SoftwareFJError):
    """Se lanza cuando un cálculo de costos produce resultados imposibles."""
    pass

print("Excepciones personalizadas definidas:")
for cls in (SoftwareFJError, DatoInvalidoError, ParametroFaltanteError,
            ServicioNoDisponibleError, ReservaInvalidaError,
            CalculoInconsistenteError):
    print(f"  - {cls.__name__}")

Excepciones personalizadas definidas:
  - SoftwareFJError
  - DatoInvalidoError
  - ParametroFaltanteError
  - ServicioNoDisponibleError
  - ReservaInvalidaError
  - CalculoInconsistenteError


## 3. Clase abstracta `EntidadBase`

Define el contrato común que toda entidad del sistema (Cliente, Servicio)
debe cumplir. Demuestra el principio de **abstracción**.

In [3]:
class EntidadBase(ABC):
    """Clase abstracta que representa una entidad genérica del sistema."""

    def __init__(self, identificador: str):
        if not identificador or not isinstance(identificador, str):
            raise DatoInvalidoError("El identificador no puede estar vacío.")
        self._id = identificador
        self._fecha_creacion = datetime.now()

    @property
    def id(self) -> str:
        return self._id

    @property
    def fecha_creacion(self) -> datetime:
        return self._fecha_creacion

    @abstractmethod
    def descripcion(self) -> str:
        """Retorna una descripción textual de la entidad."""
        pass

    @abstractmethod
    def validar(self) -> bool:
        """Valida la integridad de los datos de la entidad."""
        pass

print("Clase abstracta EntidadBase definida.")

Clase abstracta EntidadBase definida.


## 4. Clase `Cliente` – Estudiante asignado: *Cliente*

Hereda de `EntidadBase`. Aplica **encapsulación** mediante atributos privados
y *properties* con validaciones rigurosas (correo, teléfono, documento).

In [4]:
class Cliente(EntidadBase):
    """Cliente registrado en el sistema Software FJ."""

    _PATRON_EMAIL = re.compile(r"^[\w\.-]+@[\w\.-]+\.\w{2,}$")
    _PATRON_TELEFONO = re.compile(r"^\d{7,10}$")
    _PATRON_DOCUMENTO = re.compile(r"^\d{6,12}$")

    def __init__(self, identificador, nombre, documento, email, telefono):
        super().__init__(identificador)
        self.nombre = nombre
        self.documento = documento
        self.email = email
        self.telefono = telefono
        logger.info(f"Cliente creado correctamente: {self._id} - {self.__nombre}")

    @property
    def nombre(self):
        return self.__nombre

    @nombre.setter
    def nombre(self, valor):
        if not valor or len(valor.strip()) < 3:
            raise DatoInvalidoError(
                "El nombre del cliente debe tener al menos 3 caracteres.")
        self.__nombre = valor.strip()

    @property
    def documento(self):
        return self.__documento

    @documento.setter
    def documento(self, valor):
        if not self._PATRON_DOCUMENTO.match(str(valor)):
            raise DatoInvalidoError(
                f"Documento inválido '{valor}'. Debe contener entre 6 y 12 dígitos.")
        self.__documento = valor

    @property
    def email(self):
        return self.__email

    @email.setter
    def email(self, valor):
        if not self._PATRON_EMAIL.match(str(valor)):
            raise DatoInvalidoError(f"Correo electrónico inválido: '{valor}'.")
        self.__email = valor

    @property
    def telefono(self):
        return self.__telefono

    @telefono.setter
    def telefono(self, valor):
        if not self._PATRON_TELEFONO.match(str(valor)):
            raise DatoInvalidoError(
                f"Teléfono inválido '{valor}'. Debe contener entre 7 y 10 dígitos.")
        self.__telefono = valor

    def descripcion(self):
        return (f"Cliente[{self._id}] {self.__nombre} | "
                f"Doc: {self.__documento} | Email: {self.__email}")

    def validar(self):
        return bool(self.__nombre and self.__documento
                    and self.__email and self.__telefono)

print("Clase Cliente definida.")

Clase Cliente definida.


## 5. Clase abstracta `Servicio` y especializaciones – Estudiante asignado: *Servicios*

`Servicio` define el comportamiento común. Las tres subclases concretas
(`ReservaSala`, `AlquilerEquipo`, `AsesoriaTecnica`) demuestran
**herencia** y **polimorfismo** mediante `calcular_costo()` y `descripcion()`.

El método `calcular_costo_total()` ejemplifica la **sobrecarga** en Python
mediante parámetros con valores por defecto.

In [5]:
class Servicio(EntidadBase):
    """Clase abstracta para cualquier servicio ofrecido por Software FJ."""

    def __init__(self, identificador, nombre, tarifa_base, disponible=True):
        super().__init__(identificador)
        if tarifa_base <= 0:
            raise DatoInvalidoError(
                f"La tarifa base debe ser positiva (recibido: {tarifa_base}).")
        self._nombre = nombre
        self._tarifa_base = tarifa_base
        self._disponible = disponible
        # El log de creación se realiza en cada subclase, una vez que sus
        # atributos específicos hayan sido asignados.

    @property
    def nombre(self):
        return self._nombre

    @property
    def tarifa_base(self):
        return self._tarifa_base

    @property
    def disponible(self):
        return self._disponible

    @disponible.setter
    def disponible(self, valor):
        self._disponible = bool(valor)

    @abstractmethod
    def calcular_costo(self, duracion):
        """Calcula el costo del servicio según duración (polimórfico)."""
        pass

    def calcular_costo_total(self, duracion, impuesto=0.19, descuento=0.0):
        """Versión sobrecargada con impuesto y descuento opcionales."""
        if not 0 <= descuento <= 1:
            raise CalculoInconsistenteError(
                f"El descuento debe estar entre 0 y 1 (recibido: {descuento}).")
        if impuesto < 0:
            raise CalculoInconsistenteError(
                f"El impuesto no puede ser negativo (recibido: {impuesto}).")
        costo_base = self.calcular_costo(duracion)
        costo_con_descuento = costo_base * (1 - descuento)
        costo_final = costo_con_descuento * (1 + impuesto)
        if costo_final < 0:
            raise CalculoInconsistenteError("El costo final es negativo.")
        return round(costo_final, 2)

    def validar(self):
        return bool(self._nombre) and self._tarifa_base > 0


class ReservaSala(Servicio):
    """Servicio de reserva de salas por horas."""

    def __init__(self, identificador, nombre, tarifa_base,
                 capacidad, disponible=True):
        super().__init__(identificador, nombre, tarifa_base, disponible)
        if capacidad <= 0:
            raise DatoInvalidoError("La capacidad de la sala debe ser positiva.")
        self._capacidad = capacidad
        logger.info(f"Servicio creado: {self.descripcion()}")

    def calcular_costo(self, duracion):
        if duracion <= 0:
            raise DatoInvalidoError("La duración debe ser mayor que cero.")
        return self._tarifa_base * duracion

    def descripcion(self):
        return (f"[Sala {self._id}] {self._nombre} | Capacidad: "
                f"{self._capacidad} personas | Tarifa: ${self._tarifa_base}/h")


class AlquilerEquipo(Servicio):
    """Servicio de alquiler de equipos por días."""

    def __init__(self, identificador, nombre, tarifa_base,
                 stock, disponible=True):
        super().__init__(identificador, nombre, tarifa_base, disponible)
        if stock < 0:
            raise DatoInvalidoError("El stock no puede ser negativo.")
        self._stock = stock
        logger.info(f"Servicio creado: {self.descripcion()}")

    def calcular_costo(self, duracion):
        if duracion <= 0:
            raise DatoInvalidoError("Los días de alquiler deben ser positivos.")
        recargo = 1.10 if duracion > 7 else 1.0
        return self._tarifa_base * duracion * recargo

    def descripcion(self):
        return (f"[Equipo {self._id}] {self._nombre} | Stock: {self._stock} | "
                f"Tarifa: ${self._tarifa_base}/día")

    def reducir_stock(self):
        if self._stock <= 0:
            raise ServicioNoDisponibleError(
                f"No hay stock disponible para el equipo '{self._nombre}'.")
        self._stock -= 1


class AsesoriaTecnica(Servicio):
    """Servicio de asesoría especializada cobrada por hora."""

    def __init__(self, identificador, nombre, tarifa_base,
                 area_experticia, disponible=True):
        super().__init__(identificador, nombre, tarifa_base, disponible)
        if not area_experticia:
            raise ParametroFaltanteError("Debe indicar el área de experticia.")
        self._area = area_experticia
        logger.info(f"Servicio creado: {self.descripcion()}")

    def calcular_costo(self, duracion):
        if duracion <= 0:
            raise DatoInvalidoError("La duración de la asesoría debe ser > 0.")
        factor = 1.20 if duracion >= 5 else 1.0
        return self._tarifa_base * duracion * factor

    def descripcion(self):
        return (f"[Asesoría {self._id}] {self._nombre} | Área: {self._area} | "
                f"Tarifa: ${self._tarifa_base}/h")

print("Clases de Servicio definidas: ReservaSala, AlquilerEquipo, AsesoriaTecnica.")

Clases de Servicio definidas: ReservaSala, AlquilerEquipo, AsesoriaTecnica.


## 6. Clase `Reserva` – Estudiante asignado: *Reservas*

Vincula un cliente con un servicio. Implementa el **ciclo de vida** de la
reserva (PENDIENTE → CONFIRMADA → PROCESADA / CANCELADA) y demuestra el
uso de `try/except/else`, `try/except/finally` y **encadenamiento de
excepciones** con `raise ... from ...`.

In [6]:
class Reserva:
    """Reserva que vincula un cliente con un servicio."""

    ESTADOS_VALIDOS = {"PENDIENTE", "CONFIRMADA", "CANCELADA", "PROCESADA"}

    def __init__(self, identificador, cliente, servicio, duracion):
        if not isinstance(cliente, Cliente):
            raise ReservaInvalidaError("Se requiere un objeto Cliente válido.")
        if not isinstance(servicio, Servicio):
            raise ReservaInvalidaError("Se requiere un objeto Servicio válido.")
        if duracion <= 0:
            raise ReservaInvalidaError("La duración debe ser mayor que cero.")
        if not servicio.disponible:
            raise ServicioNoDisponibleError(
                f"El servicio '{servicio.nombre}' no está disponible.")
        self._id = identificador
        self._cliente = cliente
        self._servicio = servicio
        self._duracion = duracion
        self._estado = "PENDIENTE"
        self._fecha_reserva = datetime.now()
        logger.info(f"Reserva {self._id} creada para "
                    f"{cliente.nombre} -> {servicio.nombre}")

    @property
    def id(self): return self._id
    @property
    def estado(self): return self._estado
    @property
    def cliente(self): return self._cliente
    @property
    def servicio(self): return self._servicio

    def confirmar(self):
        """Confirma la reserva (uso de try/except/else)."""
        try:
            if self._estado != "PENDIENTE":
                raise ReservaInvalidaError(
                    f"No se puede confirmar reserva en estado '{self._estado}'.")
        except ReservaInvalidaError as e:
            logger.error(f"Error al confirmar reserva {self._id}: {e}")
            raise
        else:
            self._estado = "CONFIRMADA"
            logger.info(f"Reserva {self._id} confirmada.")

    def cancelar(self):
        if self._estado == "PROCESADA":
            raise ReservaInvalidaError(
                "No se puede cancelar una reserva ya procesada.")
        self._estado = "CANCELADA"
        logger.info(f"Reserva {self._id} cancelada.")

    def procesar(self, impuesto=0.19, descuento=0.0):
        """Procesa la reserva: calcula costo y actualiza estado.

        Demuestra try/except/finally + encadenamiento de excepciones."""
        costo_total = 0.0
        try:
            if self._estado != "CONFIRMADA":
                raise ReservaInvalidaError(
                    "Solo se pueden procesar reservas confirmadas.")
            costo_total = self._servicio.calcular_costo_total(
                self._duracion, impuesto, descuento)
            if isinstance(self._servicio, AlquilerEquipo):
                self._servicio.reducir_stock()
            self._estado = "PROCESADA"
        except CalculoInconsistenteError as e:
            logger.error(f"Error de cálculo en reserva {self._id}: {e}")
            # Encadenamiento: nueva excepción preservando la causa original
            raise ReservaInvalidaError(
                "No se pudo procesar la reserva por un cálculo inválido.") from e
        except SoftwareFJError as e:
            logger.error(f"Error procesando reserva {self._id}: {e}")
            raise
        finally:
            # finally se ejecuta siempre, con o sin error
            logger.info(f"Intento de procesamiento finalizado para reserva "
                        f"{self._id} | estado actual: {self._estado}")
        return costo_total

    def __str__(self):
        return (f"Reserva[{self._id}] Cliente: {self._cliente.nombre} | "
                f"Servicio: {self._servicio.nombre} | "
                f"Duración: {self._duracion} | Estado: {self._estado}")

print("Clase Reserva definida.")

Clase Reserva definida.


## 7. Gestor principal del sistema – Estudiante asignado: *Integración final*

Centraliza listas internas y operaciones de alto nivel.

In [7]:
class GestorSoftwareFJ:
    """Coordinador central del sistema (almacena en listas en memoria)."""

    def __init__(self):
        self._clientes = []
        self._servicios = []
        self._reservas = []
        logger.info("Gestor SoftwareFJ inicializado correctamente.")

    def registrar_cliente(self, cliente):
        try:
            if any(c.documento == cliente.documento for c in self._clientes):
                raise DatoInvalidoError(
                    f"Ya existe un cliente con documento {cliente.documento}.")
            self._clientes.append(cliente)
            logger.info(f"Cliente {cliente.id} registrado en el sistema.")
        except SoftwareFJError as e:
            logger.error(f"No se pudo registrar el cliente: {e}")
            raise

    def registrar_servicio(self, servicio):
        self._servicios.append(servicio)
        logger.info(f"Servicio {servicio.id} registrado en el sistema.")

    def crear_reserva(self, identificador, cliente, servicio, duracion):
        reserva = Reserva(identificador, cliente, servicio, duracion)
        self._reservas.append(reserva)
        return reserva

    def listar_clientes(self): return list(self._clientes)
    def listar_servicios(self): return list(self._servicios)
    def listar_reservas(self): return list(self._reservas)

print("Clase GestorSoftwareFJ definida.")

Clase GestorSoftwareFJ definida.


## 8. Simulación de 10+ operaciones completas

Combinación de operaciones válidas e inválidas que demuestran que el
sistema **continúa funcionando** ante errores graves, capturándolos
de forma controlada y registrándolos en el log.

In [8]:
print("=" * 70)
print("   SISTEMA SOFTWARE FJ - SIMULACIÓN DE OPERACIONES")
print("=" * 70)

gestor = GestorSoftwareFJ()
exitosos = 0
fallidos = 0

# OPERACIÓN 1: Registro de cliente VÁLIDO
print("\n[1] Registro de cliente VÁLIDO")
try:
    c1 = Cliente("CLI001", "Ana María Pérez", "1023456789",
                 "ana.perez@correo.com", "3001234567")
    gestor.registrar_cliente(c1)
    print(f"    OK -> {c1.descripcion()}")
    exitosos += 1
except SoftwareFJError as e:
    print(f"    ERROR -> {e}"); fallidos += 1

# OPERACIÓN 2: Cliente con email INVÁLIDO
print("\n[2] Registro de cliente con email INVÁLIDO")
try:
    Cliente("CLI002", "Pedro López", "1099887766",
            "correo_invalido", "3110000000")
except DatoInvalidoError as e:
    print(f"    EXCEPCIÓN CONTROLADA -> {e}"); fallidos += 1

# OPERACIÓN 3: Segundo cliente VÁLIDO
print("\n[3] Registro de segundo cliente VÁLIDO")
try:
    c2 = Cliente("CLI003", "Carlos Rodríguez", "1075432198",
                 "carlos.r@empresa.co", "3157654321")
    gestor.registrar_cliente(c2)
    print(f"    OK -> {c2.descripcion()}")
    exitosos += 1
except SoftwareFJError as e:
    print(f"    ERROR -> {e}"); fallidos += 1

# OPERACIÓN 4: Servicio con tarifa INVÁLIDA
print("\n[4] Creación de servicio con tarifa INVÁLIDA")
try:
    ReservaSala("SAL999", "Sala Fantasma", -5000, 10)
except DatoInvalidoError as e:
    print(f"    EXCEPCIÓN CONTROLADA -> {e}"); fallidos += 1

# OPERACIÓN 5: Tres servicios VÁLIDOS
print("\n[5] Creación VÁLIDA de tres servicios especializados")
try:
    sala = ReservaSala("SAL001", "Sala Ejecutiva", 50000, 12)
    equipo = AlquilerEquipo("EQU001", "Proyector 4K", 80000, stock=2)
    asesoria = AsesoriaTecnica("ASE001", "Consultoría Cloud",
                               120000, "AWS / Azure")
    for s in (sala, equipo, asesoria):
        gestor.registrar_servicio(s)
        print(f"    OK -> {s.descripcion()}")
    exitosos += 1
except SoftwareFJError as e:
    print(f"    ERROR -> {e}"); fallidos += 1

# OPERACIÓN 6: Reserva exitosa
print("\n[6] Reserva EXITOSA de sala con confirmación y procesamiento")
try:
    r1 = gestor.crear_reserva("RES001", c1, sala, duracion=3)
    r1.confirmar()
    costo = r1.procesar(impuesto=0.19, descuento=0.10)
    print(f"    OK -> {r1} | Costo total: ${costo:,.2f}")
    exitosos += 1
except SoftwareFJError as e:
    print(f"    ERROR -> {e}"); fallidos += 1

# OPERACIÓN 7: Reserva con duración INVÁLIDA
print("\n[7] Intento de reserva con duración INVÁLIDA (cero)")
try:
    gestor.crear_reserva("RES002", c2, asesoria, duracion=0)
except ReservaInvalidaError as e:
    print(f"    EXCEPCIÓN CONTROLADA -> {e}"); fallidos += 1

# OPERACIÓN 8: Asesoría con tarifa premium
print("\n[8] Reserva de asesoría con tarifa PREMIUM (5+ horas)")
try:
    r3 = gestor.crear_reserva("RES003", c2, asesoria, duracion=6)
    r3.confirmar()
    costo = r3.procesar()
    print(f"    OK -> {r3} | Costo total: ${costo:,.2f}")
    exitosos += 1
except SoftwareFJError as e:
    print(f"    ERROR -> {e}"); fallidos += 1

# OPERACIÓN 9: Procesar SIN confirmar
print("\n[9] Procesar reserva SIN confirmar (debe fallar)")
try:
    r4 = gestor.crear_reserva("RES004", c1, equipo, duracion=4)
    r4.procesar()
except ReservaInvalidaError as e:
    print(f"    EXCEPCIÓN CONTROLADA -> {e}"); fallidos += 1

# OPERACIÓN 10: Encadenamiento de excepciones
print("\n[10] Procesamiento con descuento INVÁLIDO (>100%)")
try:
    r5 = gestor.crear_reserva("RES005", c2, equipo, duracion=2)
    r5.confirmar()
    r5.procesar(descuento=1.5)
except ReservaInvalidaError as e:
    print(f"    EXCEPCIÓN CONTROLADA -> {e}")
    if e.__cause__:
        print(f"    Causa original         -> {e.__cause__}")
    fallidos += 1

# EXTRA: Cliente duplicado
print("\n[EXTRA] Intento de registro de cliente DUPLICADO")
try:
    dup = Cliente("CLI004", "Otra Persona", "1023456789",
                  "otra@correo.com", "3009998877")
    gestor.registrar_cliente(dup)
except DatoInvalidoError as e:
    print(f"    EXCEPCIÓN CONTROLADA -> {e}")

# Resumen
print("\n" + "=" * 70)
print("   RESUMEN DE LA SIMULACIÓN")
print("=" * 70)
print(f"   Operaciones exitosas: {exitosos}")
print(f"   Excepciones controladas: {fallidos}")
print(f"   Total de clientes registrados: {len(gestor.listar_clientes())}")
print(f"   Total de servicios registrados: {len(gestor.listar_servicios())}")
print(f"   Total de reservas creadas: {len(gestor.listar_reservas())}")
print("\n   El sistema permaneció ESTABLE durante toda la simulación.")
print("   Revise el archivo 'software_fj.log' para ver el registro completo.")
print("=" * 70)

   SISTEMA SOFTWARE FJ - SIMULACIÓN DE OPERACIONES

[1] Registro de cliente VÁLIDO
    OK -> Cliente[CLI001] Ana María Pérez | Doc: 1023456789 | Email: ana.perez@correo.com

[2] Registro de cliente con email INVÁLIDO
    EXCEPCIÓN CONTROLADA -> Correo electrónico inválido: 'correo_invalido'.

[3] Registro de segundo cliente VÁLIDO
    OK -> Cliente[CLI003] Carlos Rodríguez | Doc: 1075432198 | Email: carlos.r@empresa.co

[4] Creación de servicio con tarifa INVÁLIDA
    EXCEPCIÓN CONTROLADA -> La tarifa base debe ser positiva (recibido: -5000).

[5] Creación VÁLIDA de tres servicios especializados
    OK -> [Sala SAL001] Sala Ejecutiva | Capacidad: 12 personas | Tarifa: $50000/h
    OK -> [Equipo EQU001] Proyector 4K | Stock: 2 | Tarifa: $80000/día
    OK -> [Asesoría ASE001] Consultoría Cloud | Área: AWS / Azure | Tarifa: $120000/h

[6] Reserva EXITOSA de sala con confirmación y procesamiento
    OK -> Reserva[RES001] Cliente: Ana María Pérez | Servicio: Sala Ejecutiva | Duración: 3 | E

## 9. Visualización del archivo de logs

In [9]:
# Forzar el flush del buffer y mostrar el contenido del archivo de logs
for h in logger.handlers + logging.getLogger().handlers:
    h.flush()

with open("software_fj.log", "r", encoding="utf-8") as f:
    contenido = f.read()
print(contenido)

2026-05-06 13:46:47 | INFO | Gestor SoftwareFJ inicializado correctamente.
2026-05-06 13:46:47 | INFO | Cliente creado correctamente: CLI001 - Ana María Pérez
2026-05-06 13:46:47 | INFO | Cliente CLI001 registrado en el sistema.
2026-05-06 13:46:47 | INFO | Cliente creado correctamente: CLI003 - Carlos Rodríguez
2026-05-06 13:46:47 | INFO | Cliente CLI003 registrado en el sistema.
2026-05-06 13:46:47 | INFO | Servicio creado: [Sala SAL001] Sala Ejecutiva | Capacidad: 12 personas | Tarifa: $50000/h
2026-05-06 13:46:47 | INFO | Servicio creado: [Equipo EQU001] Proyector 4K | Stock: 2 | Tarifa: $80000/día
2026-05-06 13:46:47 | INFO | Servicio creado: [Asesoría ASE001] Consultoría Cloud | Área: AWS / Azure | Tarifa: $120000/h
2026-05-06 13:46:47 | INFO | Servicio SAL001 registrado en el sistema.
2026-05-06 13:46:47 | INFO | Servicio EQU001 registrado en el sistema.
2026-05-06 13:46:47 | INFO | Servicio ASE001 registrado en el sistema.
2026-05-06 13:46:47 | INFO | Reserva RES001 creada para

## 10. Conclusiones

El presente notebook implementa un **sistema integral orientado a objetos**
para la empresa Software FJ, demostrando de forma rigurosa los cuatro
pilares de la POO (abstracción, herencia, polimorfismo, encapsulación) y
un manejo profesional de excepciones (excepciones personalizadas,
`try/except/else`, `try/except/finally`, encadenamiento con `raise from`).

La simulación ejecutó **más de 10 operaciones** combinando casos válidos
e inválidos, y el sistema permaneció **estable** en todo momento, capturando
todos los errores y registrándolos en el archivo `software_fj.log`.